# 03 — Tools: How Agents Interact with the World

## What You'll Learn
- The tool execution loop — how ADK calls tools automatically
- Built-in tools: Google Search, Code Executor
- Custom function tools — wrapping any Python function
- ToolSets, BaseTool class, and agent-as-tool
- MCP (Model Context Protocol) integration

> **Prerequisites:** Notebook 02 (Agent deep dive)


## 3.1 How Tools Work in ADK — The Full Loop

Without tools, an agent is just an LLM with a system prompt — it can
only answer from its training data. Tools give agents **agency**: the
ability to act in the world (search, compute, call APIs).

### The Tool Execution Loop

```
┌──────────────────────────────────────────────────────────┐
│  1. USER sends message                                   │
│  2. ADK sends [instruction + history + user msg] to LLM  │
│  3. LLM responds with EITHER:                            │
│     a) Text → display to user, STOP                      │
│     b) Tool Call → ADK executes the tool                 │
│  4. ADK feeds tool result back to LLM as new message    │
│  5. GOTO step 2 (LLM now sees tool result)              │
└──────────────────────────────────────────────────────────┘
```

This is the **ReAct pattern** (Reasoning + Acting):
- **Reason**: LLM decides WHAT to do (which tool to call, with what args)
- **Act**: ADK executes the tool
- **Observe**: Tool result is fed back to LLM
- **Repeat**: Until LLM produces final text

### What Makes a Function a "Tool"?

ADK auto-converts Python functions to tools using these rules:
1. **Type hints** → generate the tool's parameter schema
2. **Docstring** → becomes the tool description the LLM sees
3. **Return value** → serialized and sent back to LLM

```python
def get_weather(city: str, units: str = "metric") -> dict:
    """Get current weather for a city.       ← LLM reads this!
    Args:
        city: City name                       ← LLM sees parameter descriptions
        units: 'metric' or 'imperial'
    Returns:
        dict with temperature, conditions
    """
    ...
```


In [1]:
import asyncio
from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai.types import Content, Part
print("✅ Core imports ready")
import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv())
print(f'API key: {"✅" if os.environ.get("GOOGLE_API_KEY") else "❌"}')
from google.adk.tools import google_search        # Built-in: real-time web search
from google.adk.code_executors import BuiltInCodeExecutor  # Built-in: run Python code
from google.adk.tools.base_tool import BaseTool, ToolContext  # For custom tool classes
from google.adk.tools.base_toolset import BaseToolset  # For grouping tools
print("✅ Tool imports ready")


✅ Core imports ready
API key: ✅
✅ Tool imports ready


## 3.2 Built-in Tool: Google Search

`google_search` gives your agent real-time access to the web.
The agent can search for current events, facts, and information
that's beyond its training cutoff.

### Requirements
- **Gemini 2 model** (gemini-2.5-flash or gemini-2.5-pro)
- **GOOGLE_API_KEY** set (already loaded from .env)

### How the LLM "Sees" the Tool
ADK converts the tool into a function declaration that's sent to the LLM:
```json
{
  "name": "google_search",
  "description": "Search the web for current information",
  "parameters": {"query": {"type": "string", "description": "Search query"}}
}
```
The LLM decides WHEN to call it and with WHAT query.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GOOGLE SEARCH DEMO — Agent searches the web in real time
# ═══════════════════════════════════════════════════════════════════

async def search_demo():
    # ─── Create agent with google_search tool ────────────────────
    # The tools= list registers available tools with the agent.
    # The LLM decides WHEN to use each tool based on the user's question.
    agent = Agent(
        name="researcher",
        model="gemini-2.5-flash",     # google_search requires Gemini 2
        instruction="""You are a web research assistant.
When the user asks about current events or facts, use google_search.
Always cite the source URLs in your answer.
If the question doesn't need a search, answer from your knowledge.""",
        tools=[google_search],         # ← Register the search tool
    )

    # ─── Standard session + runner setup ────────────────────────
    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="gs")
    r = Runner(agent=agent, app_name="t", session_service=svc)

    # ─── Ask a question that requires current information ───────
    m = Content(role="user", parts=[Part.from_text(
        text="Who won the most recent Nobel Prize in Physics and what was their discovery?"
      
    )])
    print("Query: Who won the most recent Nobel Prize in Physics?\n")

    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text:        # LLM text response
                    print(f"[{e.author}]: {p.text[:500]}")
                if p.function_call:  # LLM called google_search
                    print(f"  🔍 Calling: {p.function_call.name}({p.function_call.args.get('query', '')[:80]}...)")
                if p.function_response:  # Search results returned
                    print(f"  ✅ Search completed")

await search_demo()


Query: Who won the most recent Nobel Prize in Physics?

[researcher]: Here are some of the top news headlines for June 19th, 2026:

**International Relations and Middle East Tensions:**

*   A planned formal agreement signing ceremony between the United States and Iran has been postponed, with Vice President JD Vance not traveling to Switzerland as initially scheduled, citing unfinalized technical talks.
*   Controversy continues on Capitol Hill regarding a provision in a memorandum of understanding (MoU) to end the US-Israel war with Iran, which includes a potent


## 3.3 Built-in Tool: Code Executor

`BuiltInCodeExecutor` lets the agent write and run Python code.
This is powerful for: calculations, data analysis, chart generation,
CSV processing — anything that benefits from actual computation.

### How It Works
1. LLM generates Python code as a string
2. ADK executes it in a sandboxed environment
3. stdout and return value are sent back to LLM
4. LLM interprets the results and responds to user

### Safety Note
The default `BuiltInCodeExecutor` runs code locally. For production,
use a sandboxed executor (Docker, gVisor) or Vertex AI Code Executor.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CODE EXECUTOR DEMO — Agent writes and runs Python
# ═══════════════════════════════════════════════════════════════════

async def code_demo():
    # ─── BuiltInCodeExecutor is a class, not a function ───────────
    # Instantiate it and pass to the agent via code_executor= parameter.
    # This is DIFFERENT from google_search which is a plain function.
    agent = Agent(
        name="coder",
        model="gemini-2.5-flash",
        instruction="""You are a data analyst. When asked to calculate
or analyze data, use the code_executor to run Python.
Write clean code with comments. Print results clearly.""",
        code_executor=BuiltInCodeExecutor(),  # ← Enable code execution
    )

    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="cd")
    r = Runner(agent=agent, app_name="t", session_service=svc)

    m = Content(role="user", parts=[Part.from_text(
        text="Calculate the first 15 prime numbers. Then find their median, mean, and sum."
    )])
    print("Running code executor...\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text: print(f"[{e.author}]: {p.text[:400]}")
                if p.function_call: print(f" Tool CALL: 💻 {p.function_call.name}")

await code_demo()


Running code executor...

[coder]: Okay, I can help you with that! I will calculate the first 15 prime numbers and then determine their median, mean, and sum.


[coder]: Here are the results:

*   **The first 15 prime numbers are:**
    [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

*   **Their sum is:** 328

*   **Their mean is:** 21.866666666666667

*   **Their median is:** 19


## 3.4 Custom Function Tools

Any Python function can become a tool. The requirements:

1. **Type hints on ALL parameters** — ADK uses these to generate the schema
2. **Docstring** — first line = tool description, Args/Returns = parameter docs
3. **Return a JSON-serializable value** — dict, list, str, int, float, bool
4. **No `*args`, `**kwargs`** — all parameters must be explicit

### Optional: ToolContext parameter
Add `*, tool_context: ToolContext = None` to access session state,
artifacts, and memory from within your tool.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# DEFINING CUSTOM TOOLS — Plain Python functions with type hints
# ═══════════════════════════════════════════════════════════════════

from datetime import datetime

# ─── Tool 1: Weather lookup (mock implementation) ────────────────
def get_weather(city: str, units: str = "metric") -> dict:
    """Get current weather conditions for a city.

    Args:
        city: Name of the city (e.g., "London", "Tokyo")
        units: Temperature unit — "metric" (Celsius) or "imperial" (Fahrenheit)

    Returns:
        dict with temperature, conditions, humidity, wind_speed
    """
    # In production: call a real weather API (OpenWeatherMap, WeatherAPI, etc.)
    mock_data = {
        "London":    {"temp": 15, "conditions": "Cloudy",        "humidity": 72, "wind": 12},
        "Tokyo":     {"temp": 22, "conditions": "Sunny",         "humidity": 55, "wind": 8},
        "New York":  {"temp": 18, "conditions": "Partly Cloudy", "humidity": 60, "wind": 15},
        "Berlin":    {"temp": 13, "conditions": "Rainy",         "humidity": 80, "wind": 20},
    }
    # Default: return generic data for unknown cities
    result = mock_data.get(city, {"temp": 20, "conditions": "Clear", "humidity": 50, "wind": 10})
    # Convert temperature if imperial units requested
    if units == "imperial":
        result["temp"] = round(result["temp"] * 9/5 + 32)
    result["city"] = city
    result["units"] = units
    result["timestamp"] = datetime.now().isoformat()
    return result


# ─── Tool 2: Mortgage calculator ─────────────────────────────────
def calculate_mortgage(principal: float, annual_rate: float, years: int) -> dict:
    """Calculate monthly mortgage payments using the standard formula.

    Args:
        principal: Total loan amount in dollars (e.g., 350000)
        annual_rate: Annual interest rate as percentage (e.g., 6.5 = 6.5%)
        years: Loan term in years (e.g., 30)

    Returns:
        dict with monthly_payment, total_paid, total_interest
    """
    monthly_rate = (annual_rate / 100) / 12   # Convert annual % to monthly decimal
    num_payments = years * 12                  # Total number of monthly payments
    
    # Standard amortization formula: P * r(1+r)^n / ((1+r)^n - 1)
    if monthly_rate == 0:
        monthly_payment = principal / num_payments  # Edge case: 0% interest
    else:
        monthly_payment = principal * (monthly_rate * (1 + monthly_rate)**num_payments) / ((1 + monthly_rate)**num_payments - 1)
    
    total_paid = monthly_payment * num_payments
    return {
        "monthly_payment": round(monthly_payment, 2),
        "total_paid": round(total_paid, 2),
        "total_interest": round(total_paid - principal, 2),
        "principal": principal,
        "rate_pct": annual_rate,
        "years": years
    }

print("✅ Custom tools defined: get_weather, calculate_mortgage")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# USING CUSTOM TOOLS — Agent calls both weather and mortgage tools
# ═══════════════════════════════════════════════════════════════════

async def custom_tools_demo():
    agent = Agent(
        name="assistant",
        model="gemini-2.5-flash",
        instruction="""You are a helpful assistant with weather and mortgage tools.
When asked about weather, use get_weather for the relevant city.
When asked about mortgage payments, use calculate_mortgage.
Explain the results clearly.""",
        tools=[get_weather, calculate_mortgage],  # ← Both tools registered
    )

    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="ct")
    r = Runner(agent=agent, app_name="t", session_service=svc)

    # This single query triggers TWO tool calls!
    m = Content(role="user", parts=[Part.from_text(
        text="What is the weather in Tokyo? Also, I want to borrow $350,000 at 6.5% for 30 years. What will my monthly payment be?"
    )])
    print("Multi-tool query: weather + mortgage\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m):
        if e.content and e.content.parts and e.author != "user":
            for p in e.content.parts:
                if p.text: print(f"[{e.author}]: {p.text[:400]}")
                if p.function_call: print(f"  🔧 Calling: {p.function_call.name}(...)")

await custom_tools_demo()


## 3.5 Advanced: BaseTool and ToolSets

### BaseTool — For Complex Tools
When a tool needs initialization (API clients, config, async setup),
subclass `BaseTool` instead of using a plain function.

### ToolSets — Group Related Tools
`BaseToolset` groups related tools together (e.g., all finance tools,
all file operations). This improves organization and reusability.

### Agent as Tool
You can use an **entire agent as a tool**! Put it in another agent's
`tools` list. The parent agent can delegate specific tasks to the
child agent — like calling a specialist.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TOOLSET EXAMPLE — Grouping finance tools together
# ═══════════════════════════════════════════════════════════════════

class FinanceToolset(BaseToolset):
    """Groups all finance-related tools for reusable, organized access."""

    def get_tools(self):
        """Return the list of tools in this set."""
        return [calculate_mortgage]  # Add more finance tools here

# Now you can use: tools=[FinanceToolset()]
# instead of listing every tool individually.
print("✅ FinanceToolset ready — use: tools=[FinanceToolset()]")


## Tool Selection Guide

| Pattern | When to Use | Example |
|---------|-------------|--------|
| Plain function + type hints | Simple, stateless operations | `def lookup(query: str) -> dict` |
| `google_search` | Real-time web information | Current events, facts |
| `BuiltInCodeExecutor()` | Calculations, data processing | Math, CSV analysis |
| `BaseTool` subclass | Tools needing config/init | API clients with auth |
| `BaseToolset` subclass | Grouping 3+ related tools | All finance tools together |
| Agent in `tools` list | Delegation to specialists | Translator, code reviewer |

**Next:** [04_multi_agent.ipynb](./04_multi_agent.ipynb) — orchestrating multiple agents.
